# Brasileirão 2026 — quem cai?

Notebook de análise do projeto. A lógica fica em `src/brasileirao/`; aqui cada etapa é executada e explicada.

**Pergunta:** qual a probabilidade de cada clube ser rebaixado, dado o que já aconteceu no campeonato?

**Etapas:** 1) dados → 2) análise exploratória em SQL → 3) modelo de gols → 4) validação (backtest) → 5) simulação Monte Carlo → 6) conclusões e limitações.

In [1]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
pd.set_option("display.precision", 3)
from brasileirao import dados, banco, avaliacao, simulacao
from brasileirao.modelo import ModeloGols, prever_jogos

## 1. Dados
Fonte pública: [openfootball](https://github.com/openfootball/football.json). O arquivo bruto fica em `data/raw/` e nunca é editado à mão; a limpeza padroniza nomes, datas e placares e roda checagens de qualidade (20 clubes, 380 jogos, sem confronto repetido).

In [2]:
jogos = dados.carregar()
feitos = jogos[jogos["disputado"]]
print(f"{len(feitos)} jogos disputados, até a {feitos['rodada'].max()}ª rodada; faltam {(~jogos['disputado']).sum()}.")
jogos.head()

277 jogos disputados, até a 28ª rodada; faltam 103.


,jogo_id,rodada,data,mandante,visitante,gols_mandante,gols_visitante,disputado
0,1,1,2026-01-28,Atlético-MG,Palmeiras,2,2,True
1,2,1,2026-01-28,Chapecoense,Santos,4,2,True
2,3,1,2026-01-28,Corinthians,Bahia,1,2,True
3,4,1,2026-01-28,Coritiba,Bragantino,0,1,True
4,5,1,2026-01-28,Fluminense,Grêmio,2,1,True


## 2. Análise exploratória em SQL
As partidas vão para um banco SQLite. A visão `desempenho` transforma cada jogo em duas linhas (uma por clube), o que simplifica as consultas. Todas estão em `sql/02_analises.sql`.

In [3]:
con = banco.criar_banco(jogos)
banco.rodar(con, "resultados_liga")

,jogos,pct_mandante,pct_empate,pct_visitante,gols_por_jogo
0,277,46.9,27.8,25.3,2.67


Jogar em casa pesa muito: o mandante vence quase metade dos jogos. Por clube, a diferença entre casa e fora varia bastante:

In [4]:
banco.rodar(con, "casa_fora").head(8)

,clube,aprov_casa,aprov_fora,diferenca
0,Vitória,66.7,11.9,54.8
1,Grêmio,59.5,9.5,50.0
2,São Paulo,69.2,21.4,47.8
3,Fluminense,78.6,35.7,42.9
4,Vasco,54.8,20.5,34.2
5,Atlético-MG,66.7,33.3,33.3
6,Athletico-PR,73.8,42.9,31.0
7,Mirassol,51.1,23.1,28.0


In [5]:
banco.rodar(con, "classificacao")

,pos,clube,j,pts,v,e,d,gp,gc,sg
0,1,Flamengo,28,60,18,6,4,55,23,32
1,2,Palmeiras,28,57,16,9,3,47,21,26
2,3,Athletico-PR,28,49,14,7,7,43,32,11
3,4,Fluminense,28,48,13,9,6,44,36,8
4,5,Bahia,28,46,12,10,6,43,35,8
5,6,Cruzeiro,28,45,13,6,9,42,40,2
6,7,Atlético-MG,27,40,11,7,9,36,32,4
7,8,Santos,27,38,10,8,9,41,40,1
8,9,Coritiba,28,38,10,8,10,37,43,-6
9,10,Bragantino,27,36,10,6,11,33,31,2


In [6]:
banco.rodar(con, "forma_ultimos_5").head(8)

,clube,pts_5,sequencia
0,Flamengo,15,VVVVV
1,Santos,13,EVVVV
2,Bahia,12,VVVVD
3,Fluminense,10,VEVDV
4,Cruzeiro,9,VDVDV
5,Palmeiras,9,VEEVE
6,São Paulo,9,DVVDV
7,Vasco,9,DVDVV


## 3. Modelo de gols (Dixon-Coles)
Os gols de cada lado seguem uma Poisson cuja média depende do **ataque** de quem marca, da **defesa** de quem sofre e da **vantagem de casa**. A correção de Dixon & Coles ajusta os placares baixos. Os parâmetros saem por máxima verossimilhança ponderada (`scipy.optimize`).

Jogos atípicos pesam menos: o saldo de cada jogo é medido em desvios-padrão; até 1 DP vale 1, entre 1 e 2 DP vale 0,7, acima de 2 DP vale 0,4.

In [7]:
modelo = ModeloGols.ajustar(jogos)
import math
print(f"Jogar em casa multiplica os gols do mandante por {math.exp(modelo.mando):.2f} | rho Dixon-Coles: {modelo.rho:.3f}")
modelo.tabela_forcas()

Jogar em casa multiplica os gols do mandante por 1.32 | rho Dixon-Coles: -0.115


,clube,ataque,defesa,forca
0,Flamengo,0.321,-0.319,0.640
1,Palmeiras,0.119,-0.448,0.567
2,Athletico-PR,0.149,-0.095,0.244
3,Fluminense,0.201,0.008,0.193
4,Bahia,0.179,0.033,0.146
5,Cruzeiro,0.147,0.049,0.098
6,Atlético-MG,-0.019,-0.093,0.073
7,Bragantino,-0.080,-0.131,0.052
8,São Paulo,-0.097,-0.131,0.033
9,Santos,0.161,0.163,-0.002


![forças](../outputs/forcas.png)

## 4. Validação: o modelo prevê melhor que um palpite ingênuo?
**Backtest rolante:** para cada rodada *r* a partir da 12ª, o modelo é treinado só com as rodadas até *r* e prevê a *r+1*. Assim ele nunca vê o futuro. Comparo com duas referências: chutar 1/3 para cada resultado e usar a frequência histórica de vitória do mandante, empate e vitória do visitante.

Métricas (menor = melhor): **log loss** (pune errar com confiança) e **Brier** (erro quadrático das probabilidades).

In [8]:
avaliacao.backtest_rolante(jogos)

,log_loss,brier,acerto_favorito
modelo,1.064,0.648,0.446
frequência do treino,1.078,0.652,0.446
uniforme (1/3),1.099,0.667,0.446


### Testando hipóteses de modelagem
Três ideias foram testadas com o mesmo backtest, em vez de escolhidas no olho:
- **peso por recência** (meia-vida de 30, 60 ou 120 dias, ou sem decaimento);
- **peso menor para jogos atípicos**;
- **vantagem de casa própria de cada clube**.

In [9]:
linhas = []
for mv in (30, 60, 120, None):
    for atip in (True, False):
        for mpc in (False, True):
            r = avaliacao.backtest_rolante(jogos, meia_vida_dias=mv, usar_peso_atipico=atip, mando_por_clube=mpc).loc["modelo"]
            linhas.append({"meia_vida": mv or "sem decaimento", "peso_atipico": atip, "mando_por_clube": mpc, **r})
pd.DataFrame(linhas).sort_values("log_loss").head(8)

,meia_vida,peso_atipico,mando_por_clube,log_loss,brier,acerto_favorito
12,sem decaimento,True,False,1.064,0.648,0.446
13,sem decaimento,True,True,1.065,0.649,0.446
8,120,True,False,1.067,0.650,0.427
9,120,True,True,1.070,0.653,0.433
14,sem decaimento,False,False,1.070,0.652,0.459
15,sem decaimento,False,True,1.072,0.653,0.465
4,60,True,False,1.073,0.655,0.427
10,120,False,False,1.073,0.655,0.439


**O que os dados disseram:**
- o peso menor para jogos atípicos **ajuda** em todas as combinações, e entra no modelo final;
- o decaimento por recência **piora**: com só ~280 jogos e usando apenas gols (uma métrica ruidosa), descartar jogos antigos tira mais informação do que acrescenta;
- a vantagem de casa por clube **não melhora** a previsão: a diferença casa/fora da tabela SQL é, em boa parte, ruído de amostra pequena.

O ganho sobre a referência ingênua é real, mas modesto, o que é esperado em futebol, um esporte de poucos gols. A próxima versão deve trocar gols por **xG** (gols esperados), que é bem menos ruidoso.

## 5. Simulação Monte Carlo
Cada jogo que falta tem o placar sorteado da matriz de probabilidades do modelo. Somam-se pontos, vitórias, saldo e gols pró, e a tabela é ordenada pelos critérios de desempate. Repetido 50 mil vezes.

In [10]:
sim = simulacao.simular(modelo, jogos, n_sim=50_000)
res = simulacao.resumo(sim)
res[["clube", "pts_atuais", "pts_p10", "pts_mediana", "pts_p90", "titulo", "top4", "rebaixamento"]].round(1)

,clube,pts_atuais,pts_p10,pts_mediana,pts_p90,titulo,top4,rebaixamento
0,Flamengo,60,74.0,79.0,84.0,76.3,100.0,0.0
1,Palmeiras,57,70.0,75.0,80.0,23.4,99.9,0.0
2,Athletico-PR,49,60.0,65.0,70.0,0.2,77.1,0.0
3,Fluminense,48,59.0,64.0,69.0,0.1,63.9,0.0
4,Bahia,46,55.0,60.0,66.0,0.0,25.8,0.0
5,Cruzeiro,45,54.0,60.0,65.0,0.0,24.5,0.0
6,Atlético-MG,40,50.0,55.0,61.0,0.0,5.3,0.0
7,Bragantino,36,47.0,52.0,58.0,0.0,1.1,0.3
8,São Paulo,36,46.0,52.0,57.0,0.0,1.0,0.4
9,Santos,38,46.0,52.0,57.0,0.0,0.9,0.4


![risco](../outputs/risco_rebaixamento.png)

### Quantos pontos livram da queda?

In [11]:
c = simulacao.corte_rebaixamento(sim)
c[(c.permanencia_pct > 5) & (c.permanencia_pct < 99)]

,pontos,permanencia_pct,ocorrencias
21,39,11.506,20806
22,40,26.802,23827
23,41,48.273,27243
24,42,71.072,29846
25,43,87.246,32491
26,44,96.321,33623


![corte](../outputs/corte_rebaixamento.png)

### Próximos 10 jogos (por data)

In [12]:
restantes = jogos[~jogos["disputado"]].sort_values("data").head(10)
prever_jogos(modelo, restantes).drop(columns="jogo_id").round(2)

,rodada,mandante,visitante,p_mandante,p_empate,p_visitante,gols_esp_mandante,gols_esp_visitante,placar_mais_provavel,p_placar
0,21,Chapecoense,Vasco,0.37,0.27,0.36,1.45,1.44,1x1,0.13
1,21,São Paulo,Santos,0.45,0.28,0.26,1.52,1.11,1x1,0.14
2,21,Atlético-MG,Bragantino,0.42,0.32,0.26,1.22,0.91,1x1,0.15
3,29,Botafogo,Vasco,0.48,0.26,0.26,1.75,1.23,1x1,0.12
4,29,Bragantino,Mirassol,0.50,0.30,0.21,1.44,0.85,1x1,0.14
5,29,Cruzeiro,São Paulo,0.45,0.29,0.25,1.45,1.03,1x1,0.14
6,29,Internacional,Corinthians,0.37,0.36,0.27,0.99,0.81,0x0,0.18
7,29,Remo,Grêmio,0.40,0.30,0.30,1.34,1.14,1x1,0.14
8,29,Vitória,Chapecoense,0.50,0.27,0.22,1.62,1.01,1x1,0.13
9,29,Athletico-PR,Atlético-MG,0.48,0.29,0.23,1.51,0.96,1x1,0.14


## 6. Conclusões e limitações
- A briga contra a queda se concentra em poucos clubes; o gráfico de corte mostra a faixa de pontos a partir da qual a permanência fica provável.
- **Limitações:** o modelo usa só placares; não sabe de lesões, suspensões, troca de técnico, viagens ou calendário de copas. Cada jogo é independente dos outros (não há "momento" dentro da simulação). Os dados públicos podem estar algumas rodadas atrás do campeonato real.
- **Próximos passos:** incorporar xG e grandes chances por mando; calibrar as probabilidades (curva de confiabilidade); testar efeitos de descanso e desfalques com o mesmo backtest.